In [ ]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel
from qdrant_client import QdrantClient
from qdrant_client.models import SparseVector, Prefetch, Fusion,  FusionQuery
import torch

In [ ]:
from pathlib import Path

# --- 1. Path & Connection Configuration ---
BASE_DIR = Path.cwd().parent
DATASET_PATH = BASE_DIR / "data" / "processed" / "nitibench_cleaned_2026-03-17.parquet"
EMBEDDING_PATH = BASE_DIR / "data" / "processed" / "vectors_sparse_nitibench-ccl-bge-m3.parquet" # embedding

# เปลี่ยนจาก path เป็น url ของ localhost
QDRANT_URL = "http://localhost:6333" 
COLLECTION_NAME = "thai_laws_collection"
MODEL_NAME = "VISAI-AI/nitibench-ccl-human-finetuned-bge-m3"

In [ ]:
# --- 2. Initialize Client & Model ---
client = QdrantClient(url=QDRANT_URL) # เชื่อมต่อผ่าน URL

# เช็คว่ามี GPU (CUDA) หรือไม่
# ถ้ามีให้ใช้ 'cuda' ถ้าไม่มีให้ใช้ 'cpu'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_fp16 = True if device == 'cuda' else False  # fp16 ส่วนใหญ่รองรับเฉพาะบน GPU

print(f"กำลังรันโมเดลบน: {device.upper()}")

# 2. Initialize Model
model = BGEM3FlagModel(
    'VISAI-AI/nitibench-ccl-human-finetuned-bge-m3', 
    use_fp16=use_fp16, 
    batch_size=16,
    device=device 
)

In [ ]:
import pandas as pd
df = pd.read_parquet(EMBEDDING_PATH)
df.head()

In [ ]:
# --- Constants สำหรับชื่อ Vector ให้ตรงกับตอน Upsert ---
VECTOR_DENSE = "dense"
VECTOR_SPARSE = "sparse"

# โหลดโมเดล Reranker (ตัวแปรเดิม)
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True, devices='cuda') 

### !!!Warning this hybrid search function is not uptodate
please use hybrid_serach from src.rag.hybrid_retriever

In [ ]:
def hybrid_search(model,reranker ,query_text, limit=10):
    # 1. สร้าง Vectors สำหรับ Query (เหมือนเดิม)
    query_output = model.encode([query_text], return_dense=True, return_sparse=True, max_length=512)
    query_dense = query_output['dense_vecs'][0].tolist()
    
    sparse_dict = query_output['lexical_weights'][0]
    query_sparse = SparseVector(
        indices=[int(k) for k in sparse_dict.keys()],
        values=[float(v) for v in sparse_dict.values()]
    )

    # 2. ค้นหาแบบแยกสาย (ใช้ Prefetch โดยไม่มี FusionQuery หลัก)
    # เราจะดึง Candidate จากทั้งสองฝั่งมารวมกัน (Union) 
    # โดยใช้ limit ที่กว้างขึ้นเพื่อให้ Reranker มีตัวเลือกที่หลากหลาย
    retrieval_limit = limit

    search_result = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(query=query_dense, using=VECTOR_DENSE, limit=retrieval_limit),
            Prefetch(query=query_sparse, using=VECTOR_SPARSE, limit=retrieval_limit),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=retrieval_limit, 
        with_payload=True
    ).points

    if not search_result:
        return []

    # 3. Reranking ขั้นตอนที่ "แท้จริง"
    # นำ Candidate ทั้งหมดที่ได้จากทั้ง Dense และ Sparse มาคำนวณคะแนนใหม่
    pairs = [[query_text, r.payload['text']] for r in search_result]
    
    # คำนวณคะแนนด้วย BGE-Reranker (ใช้ Batch Processing จะเร็วกว่า)
    scores = reranker.compute_score(pairs, batch_size=32)

    # 4. Map คะแนนกลับและ Re-sort
    for i, r in enumerate(search_result):
        # แทนที่คะแนนเดิม (ที่อาจจะเป็น Score จาก Vector/RRF) ด้วย Rerank Score
        r.score = float(scores[i]) 

    # เรียงลำดับจากคะแนนสูงสุดไปต่ำสุด
    search_result.sort(key=lambda x: x.score, reverse=True)

    # คืนค่าตามจำนวนที่ต้องการจริงๆ
    return search_result[:limit]

In [ ]:
# --- ตัวอย่างการใช้งาน ---
results = hybrid_search(model, reranker, "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร", limit=3)
for res in results:
    print(f"Score: {res.score:.4f} | {res.payload['law_name']} มาตรา {res.payload['section_num']}")
    print(f"Text: {res.payload['text'][:150]}...\n")